# 01 · Fase 1 · Análisis exploratorio (EDA)

**Objetivo (40 min):** cargar los datos sin romperlos, entender qué contienen y detectar sus trampas **antes** de sacar conclusiones.

> **EDA** (*Exploratory Data Analysis*) = mirar los datos con curiosidad y desconfianza. ¿Qué hay? ¿Qué falta? ¿Qué no cuadra?

**Ruta de este notebook**

1. Leer un CSV (y ver cómo se rompe si se lee mal)
2. Primer vistazo: forma, columnas, tipos
3. Estadística descriptiva: media, mediana, percentiles
4. La brecha con dos definiciones
5. Las 5 trampas de estos datos
6. Explorar los microdatos sintéticos

In [1]:
# ⚙️ CELDA DE PREPARACIÓN · ejecútala siempre la primera (Shift + Enter)
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 30)   # que se vean todas las columnas
pd.set_option("display.width", 200)

URL_BASE = "https://innosoft-igualdad-sostenibilidad.vercel.app/data/csv/"


def ruta_de(nombre_fichero):
    """Devuelve dónde está un CSV del datatón.

    1. Si está en la carpeta datos/csv del repositorio, usa esa copia (sin internet).
    2. Si no (por ejemplo, en Google Colab), usa la URL de la web del datatón.
    """
    for carpeta in [Path("datos/csv"), Path("../datos/csv")]:   # rutas RELATIVAS
        if (carpeta / nombre_fichero).exists():
            return carpeta / nombre_fichero
    return URL_BASE + nombre_fichero


def leer(nombre_fichero):
    """Lee un CSV del datatón con las opciones que funcionan para los 15 ficheros."""
    return pd.read_csv(
        ruta_de(nombre_fichero),
        sep=";",               # las columnas se separan con punto y coma
        decimal=",",           # 27,5 significa 27.5
        thousands=".",         # 1.835.875 significa 1835875
        encoding="utf-8-sig",  # quita un carácter invisible (BOM) del principio
        na_values=["."],       # el INE escribe "." cuando no hay dato
    )


print("✅ Preparación lista")

✅ Preparación lista


## 1. Leer un CSV… y ver cómo se rompe

Un **CSV** es un fichero de texto donde cada línea es una fila y un carácter separa las columnas. Los CSV de este datatón:

- separan columnas con **`;`** (no con `,`)
- usan **`,`** como decimal y **`.`** como separador de miles (formato español)

Veamos qué pasa si leemos con las opciones por defecto:

In [2]:
url = ruta_de("DS-005_salario_sexo_andalucia_2024.csv")   # ruta local o URL

mal = pd.read_csv(url)            # ❌ por defecto, pandas espera comas
print(mal.shape)                  # (filas, columnas)
mal.head(2)

(3, 1)


,,,,,,,,,sexo;asalariados;salario_bruto_anual;bruto_p25;bruto_p50;bruto_p75;salario_eq_tiempo_completo;eq_p25;eq_p50;eq_p75;periodo;territorio;fuente
Hombres;1.835.875,00;21.103,00;8.896,00;18.148,00;26.757,00;27.182,00;18.461,00;22.189,00;29.542,00;2024;Andalucía;IECA · Los salarios en Andal...
Mujeres;1.649.500,00;17.093,00;5.745,00;14.038,00;22.797,00;24.945,00;17.483,00;20.481,00;27.358,00;2024;Andalucía;IECA · Los salarios en Andal...


😱 **Una sola columna** con todo pegado. pandas no sabía que el separador era `;`. Probemos solo con `sep=";"`:

In [3]:
regular = pd.read_csv(url, sep=";")
regular.dtypes.head(4)

sexo                   str
asalariados            str
salario_bruto_anual    str
bruto_p25              str
dtype: object

Ahora hay columnas, pero `salario_bruto_anual` es **texto** (`object`/`str`), porque `"21.103,00"` no parece un número. Si intentas sumarlo, fallará o, peor, **concatenará textos**.

✅ Con nuestra función `leer()` (definida en la celda de preparación) todo sale bien:

In [4]:
salario_sexo = leer("DS-005_salario_sexo_andalucia_2024.csv")
salario_sexo.dtypes.head(4)

sexo                       str
asalariados            float64
salario_bruto_anual    float64
bruto_p25              float64
dtype: object

## 2. Primer vistazo

Las 4 preguntas de siempre: **¿cuántas filas y columnas? ¿cómo se llaman? ¿de qué tipo son? ¿qué pinta tienen?**

In [5]:
print("Forma:", salario_sexo.shape)
print("Columnas:", list(salario_sexo.columns))
salario_sexo

Forma: (3, 13)
Columnas: ['sexo', 'asalariados', 'salario_bruto_anual', 'bruto_p25', 'bruto_p50', 'bruto_p75', 'salario_eq_tiempo_completo', 'eq_p25', 'eq_p50', 'eq_p75', 'periodo', 'territorio', 'fuente']


,sexo,asalariados,salario_bruto_anual,bruto_p25,bruto_p50,bruto_p75,salario_eq_tiempo_completo,eq_p25,eq_p50,eq_p75,periodo,territorio,fuente
0,Hombres,1835875.0,21103.0,8896.0,18148.0,26757.0,27182.0,18461.0,22189.0,29542.0,2024,Andalucía,IECA · Los salarios en Andalucía (MCVL) · cons...
1,Mujeres,1649500.0,17093.0,5745.0,14038.0,22797.0,24945.0,17483.0,20481.0,27358.0,2024,Andalucía,IECA · Los salarios en Andalucía (MCVL) · cons...
2,Ambos sexos,3485375.0,19205.0,7224.0,16212.0,25170.0,26123.0,17941.0,21420.0,28503.0,2024,Andalucía,IECA · Los salarios en Andalucía (MCVL) · cons...


📌 **Columnas de trazabilidad:** `periodo`, `territorio`, `fuente`. Son tu "DNI" del dato: cuando escribas una cifra en tus diapositivas, di siempre **de dónde sale, de qué año y de qué territorio**.

## 3. Estadística descriptiva

| Medida | Qué dice |
|---|---|
| **Media** | El "reparto a partes iguales". Se mueve mucho con los extremos. |
| **Mediana (P50)** | La persona del medio. Representa mejor a la persona típica. |
| **P25 / P75** | El 25 % cobra menos que P25; el 75 % cobra menos que P75. |
| **Desviación típica** | Cuánto se separan los datos de la media (dispersión). |

En salarios, la media suele ser **mayor** que la mediana: unos pocos sueldos muy altos tiran hacia arriba (distribución **asimétrica**).

In [6]:
cols = ["sexo", "salario_bruto_anual", "bruto_p25", "bruto_p50", "bruto_p75"]
salario_sexo[cols]

,sexo,salario_bruto_anual,bruto_p25,bruto_p50,bruto_p75
0,Hombres,21103.0,8896.0,18148.0,26757.0
1,Mujeres,17093.0,5745.0,14038.0,22797.0
2,Ambos sexos,19205.0,7224.0,16212.0,25170.0


## 4. La brecha, con dos definiciones

$$\text{brecha}_1 = \frac{H - M}{H} \qquad\qquad \text{brecha}_2 = \frac{H}{M} - 1$$

- **Brecha 1** (la oficial en Eurostat/INE): *qué parte del salario de los hombres falta en el de las mujeres*.
- **Brecha 2**: *cuánto más cobran los hombres que las mujeres*.

Y además podemos medir el salario de dos formas:

- **Bruto anual**: lo que se cobró en el año. Si trabajaste media jornada o 6 meses, cobras menos.
- **Equivalente a tiempo completo**: lo que se habría cobrado trabajando a jornada completa todo el año.

In [7]:
def brecha(h, m):
    return {"(H-M)/H": (h - m) / h, "H/M-1": h / m - 1}


s = salario_sexo.set_index("sexo")   # usamos 'sexo' como índice para buscar por nombre
filas = []
for col in ["salario_bruto_anual", "bruto_p50", "salario_eq_tiempo_completo", "eq_p50"]:
    resultado = brecha(s.loc["Hombres", col], s.loc["Mujeres", col])
    filas.append({"medida": col, **resultado})

tabla_brechas = pd.DataFrame(filas)
(tabla_brechas.set_index("medida") * 100).round(1)   # en %

,(H-M)/H,H/M-1
medida,,
salario_bruto_anual,19.0,23.5
bruto_p50,22.6,29.3
salario_eq_tiempo_completo,8.2,9.0
eq_p50,7.7,8.3


### 🧠 ¿Qué nos dice esto?

- La brecha **bruta** (19 %) es más del doble que la **equivalente** (8,2 %). La diferencia **no es un error**: refleja que las mujeres trabajan más a **jornada parcial** y tienen más **interrupciones** (cuidados, contratos cortos).
- La brecha en la **mediana bruta** (22,6 %) es mayor que en la media: el problema golpea más a la **mitad baja** de la distribución.
- Ninguna de las cifras demuestra por sí sola **discriminación**: hay que mirar la **composición** (edad, ocupación, sector, jornada). Eso lo haremos en la Fase 2.

## 5. Las 5 trampas de estos datos

Los datos reales nunca vienen limpios. Estas son las trampas que vas a encontrar en el catálogo. **Detectarlas y explicarlas suma puntos** en la rúbrica.

### Trampa 1 · Huecos que NO son ceros

In [8]:
temp = leer("DS-008_temporalidad_edad_sexo_andalucia_2025.csv")
print(temp.isna().sum())          # cuántos valores faltan en cada columna
temp[temp["indefinido_miles"].isna()]

sector               0
grupo_edad           0
sexo                 0
indefinido_miles     3
temporal_miles       0
total_miles          0
tasa_temporalidad    0
periodo              0
territorio           0
fuente               0
dtype: int64


,sector,grupo_edad,sexo,indefinido_miles,temporal_miles,total_miles,tasa_temporalidad,periodo,territorio,fuente
0,Público,16-19,Hombres,NaN,0.42,0.42,100.0,2025,Andalucía,IECA · EPA anual 2025 · consulta 28976
1,Público,16-19,Mujeres,NaN,0.13,0.13,100.0,2025,Andalucía,IECA · EPA anual 2025 · consulta 28976
2,Público,16-19,Ambos sexos,NaN,0.55,0.55,100.0,2025,Andalucía,IECA · EPA anual 2025 · consulta 28976


Hay 3 celdas vacías: personas de 16–19 años con contrato indefinido en el sector público. **No significa que haya cero**: la muestra de la EPA es tan pequeña que no se publica el dato.

❌ `df.fillna(0)` → te inventas un dato.
✅ Déjalo como `NaN` (pandas lo ignora en medias y sumas) y **dilo** en tu entrega.

### Trampa 2 · Filas de totales mezcladas con el detalle

In [9]:
print(temp["sexo"].unique())
print(temp["grupo_edad"].unique())
print(temp["sector"].unique())

<StringArray>
['Hombres', 'Mujeres', 'Ambos sexos']
Length: 3, dtype: str
<StringArray>
['16-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-54', '55-64', '65 y más', 'Total']
Length: 10, dtype: str
<StringArray>
['Público', 'Privado', 'Total']
Length: 3, dtype: str


In [10]:
# Si sumas todo a lo bruto, cuentas cada persona varias veces:
print("Suma 'a lo loco':", round(temp["total_miles"].sum()), "miles de asalariados")

# El total de verdad está en su propia fila
total_real = temp.query("sector == 'Total' and grupo_edad == 'Total' and sexo == 'Ambos sexos'")
print("Total real:      ", total_real["total_miles"].iloc[0], "miles de asalariados")

Suma 'a lo loco': 23925 miles de asalariados
Total real:       2990.68 miles de asalariados


👉 Antes de sumar o agrupar, **quita las filas `Total` y `Ambos sexos`** o filtra justo las que necesitas.

### Trampa 3 · Cada fichero escribe el sexo a su manera

In [11]:
d2 = leer("DS-002_salario_cnae_sexo_nacional_2022.csv")
d13 = leer("DS-013_microdatos_sinteticos_tic.csv")
print("DS-002:", d2["sexo"].unique())
print("DS-005:", salario_sexo["sexo"].unique())
print("DS-013:", d13["sexo"].unique())

DS-002: <StringArray>
['Ambos sexos', 'Mujeres', 'Varones']
Length: 3, dtype: str
DS-005: <StringArray>
['Hombres', 'Mujeres', 'Ambos sexos']
Length: 3, dtype: str
DS-013: <StringArray>
['Hombre', 'Mujer']
Length: 2, dtype: str


In [12]:
# Solución: un diccionario de traducción y .replace()
MAPA_SEXO = {"Varones": "Hombres", "Hombre": "Hombres", "Mujer": "Mujeres"}
d2["sexo"] = d2["sexo"].replace(MAPA_SEXO)
d2["sexo"].unique()

<StringArray>
['Ambos sexos', 'Mujeres', 'Hombres']
Length: 3, dtype: str

### Trampa 4 · Valores imposibles

`describe()` resume cada columna numérica. Fíjate en la fila **`min`**:

In [13]:
edad_sexo = leer("DS-006_salario_sexo_edad_nacional_2022.csv")
edad_sexo.describe().round(0)

,salario_medio,p10,p25,p50,p75,p90,periodo
count,33.0,33.0,33.0,33.0,33.0,33.0,33.0
mean,23148.0,9841.0,14097.0,19609.0,28889.0,40442.0,2022.0
std,10327.0,4508.0,6053.0,8699.0,13134.0,18540.0,0.0
min,-12437.0,-4194.0,-7261.0,-12540.0,-16970.0,-19979.0,2022.0
25%,21446.0,8796.0,14021.0,18934.0,25542.0,34249.0,2022.0
50%,25939.0,10277.0,15112.0,21651.0,33562.0,46744.0,2022.0
75%,29381.0,12574.0,17222.0,24238.0,36707.0,51945.0,2022.0
max,33191.0,15390.0,20295.0,27416.0,41786.0,59092.0,2022.0


In [ ]:
edad_sexo[edad_sexo["salario_medio"] < 0]

**Salarios negativos.** Eso no existe. Puede ser una marca de la fuente original (por ejemplo, datos de baja fiabilidad por muestra pequeña) o un error al preparar el fichero: **compruébalo en la tabla original del INE** (tabla 36834). DS-002 tiene el mismo problema en algunas actividades.

✅ Qué hacer: convertirlos en `NaN` (no en 0, ni quitarles el signo sin más), y **contarlo** en la entrega.

In [ ]:
cols_salario = ["salario_medio", "p10", "p25", "p50", "p75", "p90"]
edad_sexo[cols_salario] = edad_sexo[cols_salario].mask(edad_sexo[cols_salario] < 0)  # negativo -> NaN
edad_sexo[edad_sexo["grupo_edad"] == "Menos de 20 años"]

### Trampa 5 · Símbolos en vez de números

En DS-002, el INE escribe **`.`** cuando un dato no está disponible (secreto estadístico). Si no lo avisas (`na_values=["."]`), toda la columna se lee como texto. Nuestra función `leer()` ya lo tiene en cuenta:

In [ ]:
print(d2["salario_euros"].dtype)
print("Datos no disponibles:", d2["salario_euros"].isna().sum())

### Bonus · Las categorías no encajan entre ficheros

| Dataset | Grupos de edad |
|---|---|
| DS-003 (empleo TIC) | Menos de 30 · 30-39 · 40-49 · 50 y más |
| DS-004 (salario) | Menos de 30 · 30-39 · 40-49 · 50-59 · 60 y más |
| DS-008 (temporalidad) | 16-19 · 20-24 · 25-29 · … · 45-54 · 55-64 · 65 y más |
| DS-009 / DS-013 | 16-29 · 30-44 · 45-64 · 65 y más |

No puedes "pegar" estos ficheros por edad sin decidir cómo agrupar. Y a veces **no se puede**: eso es la **brecha de datos**, y documentarla es una respuesta válida.

## 6. Explorar los microdatos sintéticos (DS-013)

⚠️ **Son personas simuladas.** Sirven para practicar métodos, no para hablar del mercado laboral real.

Aquí cada fila es **una persona** (microdato), no una media (agregado). Eso permite muchas más preguntas.

In [ ]:
micro = leer("DS-013_microdatos_sinteticos_tic.csv")
print(micro.shape)
micro.head()

In [ ]:
# ¿Cuántas personas hay de cada categoría?
micro["sexo"].value_counts()

In [ ]:
# Proporción de cada ocupación dentro de cada sexo (normalize="columns" -> cada columna suma 1)
pd.crosstab(micro["ocupacion"], micro["sexo"], normalize="columns").round(3)

In [ ]:
# Salario por sexo: media, mediana y número de personas
micro.groupby("sexo")[["salario_bruto_anual", "salario_hora"]].agg(["mean", "median"]).round(1)

In [ ]:
# Distribución completa: un histograma por sexo
fig, ax = plt.subplots(figsize=(8, 4))
for sexo, color in [("Hombre", "#3b6fb6"), ("Mujer", "#d9822b")]:
    datos = micro.loc[micro["sexo"] == sexo, "salario_bruto_anual"]
    ax.hist(datos, bins=40, alpha=0.6, label=sexo, color=color, density=True)
ax.set_title("Distribución del salario bruto anual (datos SINTÉTICOS)")
ax.set_xlabel("€ brutos al año"); ax.set_ylabel("Densidad"); ax.legend()
plt.show()

### 🧠 Lo que hemos visto

- La distribución tiene **cola larga a la derecha** → media > mediana.
- Las mujeres están **sobrerrepresentadas en algunas ocupaciones** y casi ausentes en otras (*segregación ocupacional*).
- Hay más **jornada parcial** entre mujeres. ¿Cuánto de la brecha anual se debe a eso? → Fase 2.

## ✅ Checklist de la Fase 1

- [ ] Sé leer cualquier CSV del catálogo sin que se rompa.
- [ ] He mirado `shape`, `columns`, `dtypes`, `head()` y `describe()`.
- [ ] He contado los `NaN` y **no** los he convertido en ceros.
- [ ] He quitado filas de totales antes de agrupar.
- [ ] He unificado etiquetas (`Varones` → `Hombres`).
- [ ] He buscado valores imposibles.
- [ ] Tengo **una pregunta** para la Fase 2.

## ✏️ Ejercicios

1. Con **DS-015**, ¿en qué provincia es mayor la tasa de temporalidad de las mujeres? ¿Y la diferencia mujeres − hombres?
2. Con **DS-004**, compara el salario bruto y el equivalente por edad. ¿En qué grupo es mayor la diferencia? ¿Por qué crees?
3. Con **DS-013**, calcula el porcentaje de personas con jornada parcial por sexo (`pd.crosstab(..., normalize="index")`).

<details><summary>Pistas</summary>

```python
prov = leer("DS-015_temporalidad_provincia_sexo_andalucia_2025.csv")
tabla = prov.pivot(index="provincia", columns="sexo", values="tasa_temporalidad")
tabla["dif_pp"] = tabla["Mujeres"] - tabla["Hombres"]
tabla.sort_values("dif_pp", ascending=False)
```
</details>

In [ ]:
# Tu turno 👇